In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd

# Load saved data
df = spark.table('northwell_dev.omop.cdm_table_row_counts').toPandas()
df['snapshot_ts'] = pd.to_datetime(df['snapshot_ts'])

# Identify tables that are 0 across ALL snapshots
always_zero = (
    df.groupby(['schema', 'table'])['row_count']
    .max()
    .reset_index()
    .query('row_count == 0')
)
always_zero_set = set(zip(always_zero['schema'], always_zero['table']))

# Split into non-zero (to graph) and always-zero (to list)
df_nonzero = df[~df.apply(lambda r: (r['schema'], r['table']) in always_zero_set, axis=1)]

schemas = sorted(df['schema'].unique())

# --- Plot one chart per schema ---
for schema in schemas:
    schema_df = df_nonzero[df_nonzero['schema'] == schema].copy()
    if schema_df.empty:
        continue

    tables = sorted(schema_df['table'].unique())
    fig, ax = plt.subplots(figsize=(14, max(5, len(tables) * 0.25)))

    for tbl in tables:
        tbl_df = schema_df[schema_df['table'] == tbl].sort_values('snapshot_ts')
        ax.plot(tbl_df['snapshot_ts'], tbl_df['row_count'], marker='o', markersize=4, label=tbl)

    ax.set_title(f'Row Counts Over Time \u2014 {schema}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Snapshot')
    ax.set_ylabel('Row Count')
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7, ncol=1)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# --- Display always-empty tables per schema ---
print('\n' + '='*60)
print('TABLES WITH 0 ROWS ACROSS ALL SNAPSHOTS')
print('='*60)
for schema in schemas:
    empty_tables = sorted(always_zero[always_zero['schema'] == schema]['table'].tolist())
    if empty_tables:
        print(f'\n{schema} ({len(empty_tables)} empty tables):')
        for t in empty_tables:
            print(f'  \u2022 {t}')
    else:
        print(f'\n{schema}: no always-empty tables')